In [1]:
#Kneser Ney Model for Quadgram


In [2]:
path="/home/deepakchalla/Desktop/Desktop/NLP/Lab1/train.parquet"

In [3]:
from read_data import get_data 
data=get_data(path,n=10_000)

In [4]:
d=0.75 #Delta Parameter

In [5]:
from collections import defaultdict
def get_counts(data,n=4):
  counts={i:defaultdict(int) for i in range(1,n+1)}
  for sent in data:
    for i in range(1,n+1):
      tokens=["<s>"]*(i-1)+sent.split()+["</s>"]*(i-1)
      for j in range(len(tokens)-i+1):
        n_gram=tuple(tokens[j:j+i])
        counts[i][n_gram]+=1
  return counts


In [6]:
counts=get_counts(data)

In [7]:
def get_lambda(history):
  order=len(history)+1
  count=0
  if order>=4:
    return 0
  for key in counts[order].keys():
    hist=key[:-1]
    if hist==history:
      count+=1
  return count

In [8]:
V=len(counts[1])

In [9]:
def kneser_ney(w,h):
  order=1+len(h)
  if order==1:
     count=0
     for key in counts[2].keys():
       if key[-1]==w:
         count+=1
     return count/V
  den=counts[order-1].get(h,0)
  if den==0:
    return 0
  return (max(counts[order].get(h+(w,),0)-d,0)/(den))+get_lambda(h)*(d/(den))*kneser_ney(w,h[1:])

In [10]:
from math import log10 as log
def predict(sent,order=4):
  tokens=["<s>"]*(order-1)+sent.split()+["</s>"]*(order-1)
  log_prob=0
  for i in range(len(tokens)-order+1):
    history=tuple(tokens[i:i+order-1])
    word=tokens[i+order-1]
    prob=kneser_ney(word,history)
    log_prob+=log(prob) if prob>0 else 0
  return log_prob


In [11]:
test_path="/home/deepakchalla/Desktop/Desktop/NLP/Lab1/test.parquet"

In [12]:
test_data=get_data(test_path,n=1_000)